# exp02 — gradient-norm / clipping diagnostic for exp01 (learning_rate = 0, no weight updates)

Measures the pre-clip gradient norm the exp01 Trainer saw, to test whether the 8x gradient-accumulation
loss scaling (transformers 5.20.2, `*ForSequenceClassification` drops `num_items_in_batch`) changed
gradient clipping. Uses the exp01 model setup (4-bit NF4, `prepare_model_for_kbit_training`,
r=8 q/v LoRA, `modules_to_save=["score"]`, fp16, gradient checkpointing, bs 2 x GA 8, seed 42,
`max_grad_norm=1.0`) but **`learning_rate=0.0`**, so no parameter ever changes (asserted).

Runs, each `N_STEPS` optimizer steps on exp01's own first train batches (same seed/sampler):
* `step0/default`  — exp01's initial weights (seed-42 head, LoRA B=0), Trainer defaults (= exp01)
* `step0/fixed`    — same, with `trainer.model_accepts_loss_kwargs = False` (Trainer divides by GA)
* `final/default`, `final/fixed` — exp01's trained adapter + head (`is_trainable=True`, lr 0)

A callback records the post-clip gradient norm of the `score` head vs the LoRA matrices.


In [ ]:
import glob, subprocess, sys
wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes-*.whl", recursive=True))
assert wheels, "attach engs108-offline-wheels"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheels[-1]], check=True)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import gc, json, time, importlib.metadata as metadata
import numpy as np, pandas as pd, torch
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorWithPadding, Trainer, TrainerCallback, TrainingArguments, set_seed)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training

RUN_START = time.time()
DATA_DIR = "/kaggle/input/competitions/name-that-llm"
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b-base/1"
OUTPUT_DIR = "/kaggle/working"
SEED, MAX_LENGTH, VAL_SIZE = 42, 512, 0.15
TRAIN_BATCH_SIZE, GRAD_ACCUM_STEPS, MAX_GRAD_NORM = 2, 8, 1.0
N_STEPS = 12
ADAPTER_DIR = os.path.dirname(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)[0])
versions = {p: metadata.version(p) for p in ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]}
print(versions, torch.cuda.device_count())

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
with open(f"{ADAPTER_DIR}/label_map.json") as f:
    id2label = {int(k): v for k, v in json.load(f)["id2label"].items()}
label2id = {v: k for k, v in id2label.items()}
assert [id2label[i] for i in range(18)] == sorted(train["model"].unique())
df = train[["id", "prompt_id", "text", "model"]].copy()
df["labels"] = df["model"].map(label2id).astype(int)
train_idx, val_idx = next(GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED).split(df, groups=df["prompt_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True); val_df = df.iloc[val_idx].reset_index(drop=True)
assert (len(train_df), train_df["prompt_id"].nunique(), len(val_df), val_df["prompt_id"].nunique()) == (2285, 127, 414, 23)
assert not set(train_df["prompt_id"]) & set(val_df["prompt_id"])
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
train_tok = train_ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH), batched=True, remove_columns=["text"])
collator = DataCollatorWithPadding(tokenizer=tokenizer)
qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)

In [ ]:
def build(init):
    set_seed(SEED)  # exp01: set_seed immediately before from_pretrained
    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH, num_labels=18, id2label=id2label, label2id=label2id, quantization_config=qcfg,
        device_map={"": 0}, torch_dtype=torch.float16, local_files_only=True)
    m.config.pad_token_id = tokenizer.pad_token_id
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    if init == "step0":
        m = get_peft_model(m, LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.05,
                                         target_modules=["q_proj", "v_proj"], modules_to_save=["score"], bias="none"))
    else:
        m = PeftModel.from_pretrained(m, ADAPTER_DIR, is_trainable=True)
    return m


class GroupNorms(TrainerCallback):
    # Post-clip norms; clipping scales every grad by the same factor, so the head/LoRA ratio is pre-clip too.
    def __init__(self, model): self.model, self.rows = model, []
    def on_pre_optimizer_step(self, args, state, control, **kw):
        sq = {"score": 0.0, "lora": 0.0}
        for n, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                sq["score" if "score" in n else "lora"] += float(p.grad.detach().float().pow(2).sum())
        self.rows.append({k: v ** 0.5 for k, v in sq.items()})


def snapshot(m):
    return {n: p.detach().clone().cpu() for n, p in m.named_parameters() if p.requires_grad}


def measure(model, variant):
    args = TrainingArguments(output_dir="/tmp/gn", per_device_train_batch_size=TRAIN_BATCH_SIZE,
                             gradient_accumulation_steps=GRAD_ACCUM_STEPS, learning_rate=0.0, max_steps=N_STEPS,
                             fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
                             logging_steps=1, label_names=["labels"], report_to="none", seed=SEED, save_strategy="no",
                             eval_strategy="no", max_grad_norm=MAX_GRAD_NORM)
    cb = GroupNorms(model)
    tr = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=collator, callbacks=[cb])
    if variant == "fixed":
        tr.model_accepts_loss_kwargs = False
    t0 = time.time(); tr.train()
    logs = [h for h in tr.state.log_history if "grad_norm" in h]
    out = {"variant": variant, "model_accepts_loss_kwargs": bool(tr.model_accepts_loss_kwargs),
           "logged_loss": [h["loss"] for h in logs], "grad_norm_preclip": [h["grad_norm"] for h in logs],
           "post_clip_group_norms": cb.rows, "minutes": (time.time() - t0) / 60}
    print(variant, json.dumps({k: (np.round(v, 3).tolist() if isinstance(v, list) and v and not isinstance(v[0], dict) else v)
                               for k, v in out.items() if k != "post_clip_group_norms"}))
    return out

In [ ]:
results = {}
for init in ["step0", "final"]:
    model = build(init)
    model.print_trainable_parameters()
    before = snapshot(model)
    results[init] = [measure(model, "default"), measure(model, "fixed")]
    after = snapshot(model)
    assert all(torch.equal(before[n], after[n]) for n in before), "weights changed with lr=0"
    print(init, ": all", len(before), "trainable tensors unchanged")
    del model, before, after; gc.collect(); torch.cuda.empty_cache()

In [ ]:
summary = {}
for init, (d, f) in results.items():
    gd, gf = np.array(d["grad_norm_preclip"], float), np.array(f["grad_norm_preclip"], float)
    ok = np.isfinite(gd) & np.isfinite(gf)
    head = np.array([r["score"] for r in f["post_clip_group_norms"]]); lora = np.array([r["lora"] for r in f["post_clip_group_norms"]])
    summary[init] = {"default_over_fixed_grad_norm_median": float(np.median(gd[ok] / gf[ok])),
                     "true_grad_norm_median": float(np.median(gf[ok])), "true_grad_norm_min": float(gf[ok].min()),
                     "true_grad_norm_max": float(gf[ok].max()),
                     "frac_clipped_default": float((gd[ok] > MAX_GRAD_NORM).mean()),
                     "frac_clipped_fixed": float((gf[ok] > MAX_GRAD_NORM).mean()),
                     "head_over_lora_grad_norm_median": float(np.median(head / lora)),
                     "n_nonfinite_steps": int((~ok).sum())}
print(json.dumps(summary, indent=1))
with open(f"{OUTPUT_DIR}/grad_norm_check.json", "w") as fh:
    json.dump({"lr": 0.0, "n_steps": N_STEPS, "max_grad_norm": MAX_GRAD_NORM, "versions": versions,
               "summary": summary, "runs": results, "runtime_min": (time.time() - RUN_START) / 60}, fh, indent=1)